In [1]:
import sys
import warnings
sys.path.append('..')


import pandas as pd
import numpy as np
from catboost import CatBoostRegressor

from src.config import TRAIN_PATH, TEST_PATH, RANDOM_SEED
from src.preprocessing import HousePricesPreprocessor
from src.validation import cross_validate
from src.features import add_features

warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

In [3]:
y = np.log1p(train['SalePrice'])

# Выбросы (до preprocessing!)
outlier_idx = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 200000)].index
train = train.drop(outlier_idx)
y = y.drop(outlier_idx)

train = train.drop(['SalePrice', 'Id'], axis=1)
test = test.drop('Id', axis=1)

print(f"Train: {train.shape}, Test: {test.shape}")

Train: (1458, 79), Test: (1459, 79)


In [4]:
train = add_features(train)
test = add_features(test)

print(f"Train: {train.shape}")
print(f"Новые фичи: {train[['TotalSF', 'HouseAge', 'RemodAge', 'TotalBath', 'TotalPorch']].describe().round(1)}")

Train: (1458, 87)
Новые фичи:        TotalSF  HouseAge  RemodAge  TotalBath  TotalPorch
count   1458.0    1458.0    1458.0     1458.0      1458.0
mean    2557.2      36.6      23.0        2.2        86.7
std      774.1      30.2      20.6        0.8       104.8
min      334.0       0.0       0.0        1.0         0.0
25%     2008.5       8.0       4.0        2.0         0.0
50%     2473.0      35.0      14.0        2.0        48.0
75%     3002.2      54.0      41.0        2.5       135.8
max     6872.0     136.0      60.0        6.0      1027.0


In [5]:
train_base = pd.read_csv(TRAIN_PATH)
test_base = pd.read_csv(TEST_PATH)
y_base = np.log1p(train_base['SalePrice'])

outlier_idx_base = train_base[(train_base['GrLivArea'] > 4000) & (train_base['SalePrice'] < 200000)].index
train_base = train_base.drop(outlier_idx_base)
y_base = y_base.drop(outlier_idx_base)
train_base = train_base.drop(['SalePrice', 'Id'], axis=1)

model = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=6, random_seed=RANDOM_SEED, verbose=0)

БЕЗ новых фичей

In [6]:
prep_base = HousePricesPreprocessor(scale=False)
X_base = prep_base.fit_transform(train_base, np.expm1(y_base))

scores_before = cross_validate(model, X_base, y_base)

RMSE по фолдам: [0.1095 0.1291 0.1041 0.1249 0.1183]
Среднее: 0.1172 ± 0.0093


С новыми фичами

In [7]:
prep_new = HousePricesPreprocessor(scale=False)
X_new = prep_new.fit_transform(train, np.expm1(y))

scores_after = cross_validate(model, X_new, y)

RMSE по фолдам: [0.1092 0.127  0.1037 0.1232 0.1196]
Среднее: 0.1165 ± 0.0087


In [8]:
print(f"Разница: {scores_before.mean():.4f} → {scores_after.mean():.4f} ({scores_after.mean() - scores_before.mean():+.4f})")

Разница: 0.1172 → 0.1165 (-0.0007)
